In [1]:
import tgt
#from tgt import read_textgrid, write_textgrid
from tgt.core import IntervalTier, TextGrid
import os
import pandas as pd

In [2]:
def merge_short_intervals(input_path, output_path, min_duration=3.5, max_duration=9.0):
    tg = tgt.read_textgrid(input_path)
    tier = tg.get_tier_by_name('текст')
    new_tier = IntervalTier(name=tier.name, start_time=tier.start_time, end_time=tier.end_time)
    current_text = ""
    current_start = None
    current_end = None
    for interval in tier: #пустые интервалы пропускаем
        if interval.text.strip() == "":
            continue 
        duration = interval.end_time - interval.start_time
        if current_start is None:
            current_start = interval.start_time
            current_end = interval.end_time
            current_text = interval.text
        else:                    #проверим, можно ли объединить
            if (current_end - current_start) + duration <= max_duration:
                current_end = interval.end_time
                current_text += " " + interval.text
            else:               #если подходит по длительности, пропускаем
                if (current_end - current_start) >= min_duration:
                    new_tier.add_interval(tgt.core.Interval(current_start, current_end, current_text))
                else:
                    pass
                current_start = interval.start_time
                current_end = interval.end_time
                current_text = interval.text
    if current_start is not None and (current_end - current_start) >= min_duration:
        new_tier.add_interval(tgt.core.Interval(current_start, current_end, current_text))
    new_tg = TextGrid()
    new_tg.add_tier(new_tier)
    tgt.io.write_to_file(new_tg, output_path, format="long")

In [4]:
#так как руками я поделила евангелие на слишком маленькие части, попробуем их объединить кодом
def merge_all_verses(tgt_path):
    for file in os.listdir(tgt_path):
        if not file.endswith('_new.TextGrid') and file.endswith('.TextGrid'):
            file_path = os.path.join(tgt_path, file)
            merge_short_intervals(file_path, file_path.replace('.TextGrid', '_new.TextGrid'))

In [5]:
merge_all_verses('/Users/air/Downloads/Ненецкие аудио/Библейские ненецкие аудио/Mark/Mark old tgt')
merge_all_verses('/Users/air/Downloads/Ненецкие аудио/Библейские ненецкие аудио/John/John old tgt')

In [6]:
def pd_from_tgt(path): #возьмём функции из препроцессинга и посмотрим теперь на статистику файлов
    data = []
    for file in os.listdir(path):
        if file.endswith('.TextGrid'):
            tg = tgt.io.read_textgrid(os.path.join(path, file))
            tier = tg.get_tier_by_name('текст')
            for i, interval in enumerate(tier):
                    data.append({
                        "file_name" : os.path.join(path.replace('tgt', 'wav'), file.replace('_new.TextGrid', '.wav')),
                        "start_time": interval.start_time,
                        "end_time": interval.end_time,
                        "label": interval.text,
                        "length": interval.end_time - interval.start_time
                    })
    df = pd.DataFrame(data)
    df.to_csv(path.replace(' tgt', '.csv'), index=False)
    return df

In [7]:
def calculate_time(dataframe):
    print(f'Средняя длина интервала {dataframe['length'].mean()}, максимальная {dataframe['length'].max()}, минимальная {dataframe['length'].min()}')
    print(f'Звучащее время {dataframe['length'].sum() // 3600}ч {dataframe['length'].sum() % 3600 // 60}мин {dataframe['length'].sum() % 60}с')

In [10]:
new_john_df = pd_from_tgt('/Users/air/Downloads/Ненецкие аудио/Библейские ненецкие аудио/John/John tgt')
calculate_time(new_john_df)

Средняя длина интервала 7.373188234636689, максимальная 10.96875, минимальная 3.5100000000000007
Звучащее время 1.0ч 57.0мин 58.260705251221225с


In [11]:
new_john_df[:10]

,file_name,start_time,end_time,label,length
0,/Users/air/Downloads/Ненецкие аудио/Библейски...,2.241594,7.465352,"иисус миманда сер’, соя” манда мальӈгэхэд сэвс...",5.223758
1,/Users/air/Downloads/Ненецкие аудио/Библейски...,7.750168,15.572844,"«равви, тюку ненэць’ ӈамгэ сэвси” соявы? ханяӈ...",7.822676
2,/Users/air/Downloads/Ненецкие аудио/Библейски...,16.298469,26.001594,» иисус ма: «тарця ӈэвада тикы хасава’ хэбяхах...,9.703125
3,/Users/air/Downloads/Ненецкие аудио/Библейски...,26.558469,36.042219,ялянда ныклавдавэй’ си”ми ӈэдаравы нум’ серо с...,9.483750
4,/Users/air/Downloads/Ненецкие аудио/Библейски...,36.615969,46.639719,тикы вадида хэт”махаданда ян’ сабци”. сабцямда...,10.023750
5,/Users/air/Downloads/Ненецкие аудио/Библейски...,47.179719,53.507844,"«силоамам’ нюбета халтаӈгось’ ян’ хань”, сэвад...",6.328125
6,/Users/air/Downloads/Ненецкие аудио/Библейски...,54.149094,58.447973,"тиканда пыда хая, сэвда халта. таняд сэвсавэйӈ...",4.298879
7,/Users/air/Downloads/Ненецкие аудио/Библейски...,58.739760,66.062844,"тёняӈы нида, ӈахат сита ӈэванаӈэ манэ” мы” тар...",7.323083
8,/Users/air/Downloads/Ненецкие аудио/Библейски...,66.687219,73.099719,» ханяӈыдо’ тарем’ ма”: «пыданё” ». ӈани ханяӈ...,6.412500
9,/Users/air/Downloads/Ненецкие аудио/Библейски...,73.622844,79.225344,пыда ӈо” маси”: «мань ӈэ”нидам’ ». тадтикахад ...,5.602500


In [12]:
new_mark_df = pd_from_tgt('/Users/air/Downloads/Ненецкие аудио/Библейские ненецкие аудио/Mark/Mark tgt')
calculate_time(new_mark_df)

Средняя длина интервала 7.778638217601607, максимальная 10.850625000000008, минимальная 3.5774999999999864
Звучащее время 1.0ч 29.0мин 58.37492301551538с


In [13]:
new_mark_df[:10]

,file_name,start_time,end_time,label,length
0,/Users/air/Downloads/Ненецкие аудио/Библейски...,3.422844,8.535969,"иисус няндо’ ма: «ненэся нянда” мадм’, тюкохон...",5.113125
1,/Users/air/Downloads/Ненецкие аудио/Библейски...,9.126594,13.834719,ханяӈы” ненэця” нум’ параӈода’ я ӈарка ныхым’ ...,4.708125
2,/Users/air/Downloads/Ненецкие аудио/Библейски...,15.606594,23.925969,"мат” яля’ ваера”махад, иисус тохоламбадахатата...",8.319375
3,/Users/air/Downloads/Ненецкие аудио/Библейски...,25.022844,33.544719,ӈаво” пирка”на сэвто’ сыртан’ пыда мирбяда ӈан...,8.521875
4,/Users/air/Downloads/Ненецкие аудио/Библейски...,34.827219,41.262711,сэв” пэсьда”. я’ сяр’ ниня хибяхарт тикахад ва...,6.435492
5,/Users/air/Downloads/Ненецкие аудио/Библейски...,42.245029,49.896594,"тикы’ пуд няндо’ илия, моисей ӈадимяха’, иисус...",7.651564
6,/Users/air/Downloads/Ненецкие аудио/Библейски...,50.487219,59.987844,"петр ӈани’ иисусан’ ма: «равви*, тюкохона мэва...",9.500625
7,/Users/air/Downloads/Ненецкие аудио/Библейски...,60.696594,70.433469,"нябиюм’ моисей’ ед’ ӈэӈгу, ӈани ӈопойхав илия’...",9.736875
8,/Users/air/Downloads/Ненецкие аудио/Библейски...,71.429094,78.010344,"ӈаво” пирка”на тир ӈадимя, сиддо’ тикы тир тид...",6.581250
9,/Users/air/Downloads/Ненецкие аудио/Библейски...,78.854094,87.139719,"«тикы мань сянда нюми, муӈганда намдорӈада”». ...",8.285625


In [14]:
new_mark_df.to_csv('mark.csv', index=False)
new_john_df.to_csv('john.csv', index=False)